In [ ]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Install packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/drive/My Drive/FINAL YEAR PROJECT/Data/tea_with_usd.csv')

In [ ]:
# Pick a random sample
random_rows = df.sample(n=500000, random_state=42)
random_rows.head()

,Factory,Elevation,SubElevation,ManagementType,processingMethod,Name,ATCRegion,Grade,Sale_Type,NetWeight,Price,NetValue,SaleDate,SaleMonth,ExchangeRate
64804,MF1188,Low,LOW,PRIVATE,Orthodox,BALANGODA,RATNAPURA,FBOPF1,Auction Sale,480.0,730.0,350400.0,2021-01-12,1.0,187
950533,MF1476,Low,LOW,PRIVATE,Orthodox,KALUTARA,MATHUGAMA,FBOPFSP,Auction Sale,150.0,1000.0,150000.0,2020-06-02,6.0,186
2290428,MF1362,Low,LOW,PRIVATE,Orthodox,DENIYAYA,MATARA,BP,Auction Sale,440.0,450.0,198000.0,2020-12-16,12.0,188
2193771,MF0835,Medium,WESTERN-MEDIUM,PRIVATE,Orthodox,KOTMALE,NUWARA ELIYA,PEKOE1,Auction Sale,800.0,920.0,736000.0,2020-12-02,12.0,186
1220284,MF1417,Low,LOW,PRIVATE,Orthodox,MORAWAKE,MATARA,OPA,Auction Sale,520.0,1550.0,806000.0,2022-07-06,7.0,360


In [ ]:
# Drop unnecessary columns
random_rows = random_rows.drop(columns=['Sale_Type'])
random_rows = random_rows.drop(columns=['ManagementType'])
random_rows = random_rows.drop(columns=['Factory'])

# Rename 'Name' to 'SubRegion' for clarity
random_rows = random_rows.rename(columns={'Name': 'SubRegion'})

# Drop all missing values
random_rows = random_rows.dropna()

In [ ]:
random_rows['SaleDate'] = pd.to_datetime(random_rows['SaleDate'])

# Convert SaleDate to year and month features
random_rows['SaleYear'] = random_rows['SaleDate'].dt.year
random_rows['SaleMonth'] = random_rows['SaleDate'].dt.month

# Drop the original SaleDate column since it's not numerical
random_rows = random_rows.drop(columns=['SaleDate'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

In [ ]:
# Numeric features
numeric_columns = ['NetWeight', 'ExchangeRate', 'SaleMonth', 'SaleYear']

In [ ]:
# categorical columns to encode
categorical_columns = ['ATCRegion', 'Grade', 'Elevation', 'processingMethod', 'SubRegion', 'SubElevation']

# one-hot encoding
df_encoded = pd.get_dummies(random_rows, columns=categorical_columns, drop_first=True)

In [ ]:
# Combine numeric and categorical features
features_final = numeric_columns + [col for col in df_encoded.columns if col not in numeric_columns + ['NetValue']]

In [ ]:
# Define final features and target variable
features_final = list(df_encoded.columns)
features_final.remove('NetValue')

# Prepare final feature set and target variable
X_final = df_encoded[features_final]
y_final = df_encoded['NetValue']

In [ ]:
# Split data into training and testing sets
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

# Train the improved Random Forest model
rf_model_final = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=15, min_samples_split=5)
rf_model_final.fit(X_train_final, y_train_final)

# Predict on test set
y_pred_rf_final = rf_model_final.predict(X_test_final)

In [ ]:
y_fitted_rf_final = rf_model_final.predict(X_train_final)

In [ ]:
# Evaluate training performance
mae_rf_training = mean_absolute_error(y_train_final, y_fitted_rf_final)
r2_rf_training = r2_score(y_train_final, y_fitted_rf_final)
mape_rf_training = mean_absolute_percentage_error(y_train_final, y_fitted_rf_final)

mae_rf_training, r2_rf_training, mape_rf_training

(312.5627759712459, 0.9972617665804995, 0.0007143991776463474)

In [ ]:
# Evaluate improved model performance
mae_rf_final = mean_absolute_error(y_test_final, y_pred_rf_final)
r2_rf_final = r2_score(y_test_final, y_pred_rf_final)
mape_rf_final = mean_absolute_percentage_error(y_test_final, y_pred_rf_final)

mae_rf_final, r2_rf_final, mape_rf_final

(420.10248625414334, 0.9999388683195515, 0.0007869009210292356)

In [ ]:
import joblib
joblib.dump(rf_model_final, 'random_forest_regression_model_V3.pkl')

['random_forest_regression_model_V2.pkl']

In [ ]:
model_features = list(X_train_final.columns)

# Save the feature list to a file for reuse
with open('model_features.txt', 'w') as f:
    for feature in model_features:
        f.write(f"{feature}\n")

TESTING WITH UNSEEN DATA

In [ ]:
unseen_rows = df.sample(n=1000000, random_state=42)
# unseen_rows = df.copy()
unseen_rows.head()

,Factory,Elevation,SubElevation,ManagementType,processingMethod,Name,ATCRegion,Grade,Sale_Type,NetWeight,Price,NetValue,SaleDate,SaleMonth,ExchangeRate
64804,MF1188,Low,LOW,PRIVATE,Orthodox,BALANGODA,RATNAPURA,FBOPF1,Auction Sale,480.0,730.0,350400.0,2021-01-12,1.0,187
950533,MF1476,Low,LOW,PRIVATE,Orthodox,KALUTARA,MATHUGAMA,FBOPFSP,Auction Sale,150.0,1000.0,150000.0,2020-06-02,6.0,186
2290428,MF1362,Low,LOW,PRIVATE,Orthodox,DENIYAYA,MATARA,BP,Auction Sale,440.0,450.0,198000.0,2020-12-16,12.0,188
2193771,MF0835,Medium,WESTERN-MEDIUM,PRIVATE,Orthodox,KOTMALE,NUWARA ELIYA,PEKOE1,Auction Sale,800.0,920.0,736000.0,2020-12-02,12.0,186
1220284,MF1417,Low,LOW,PRIVATE,Orthodox,MORAWAKE,MATARA,OPA,Auction Sale,520.0,1550.0,806000.0,2022-07-06,7.0,360


In [ ]:
# Drop unnecessary columns
unseen_rows = unseen_rows.drop(columns=['Sale_Type'])
unseen_rows = unseen_rows.drop(columns=['ManagementType'])
unseen_rows = unseen_rows.drop(columns=['Factory'])

# Rename 'Name' to 'SubRegion' for clarity
unseen_rows = unseen_rows.rename(columns={'Name': 'SubRegion'})

# Drop all missing values
unseen_rows = unseen_rows.dropna()

In [ ]:
unseen_rows['SaleDate'] = pd.to_datetime(unseen_rows['SaleDate'])

# Convert SaleDate to year and month features
unseen_rows['SaleYear'] = unseen_rows['SaleDate'].dt.year
unseen_rows['SaleMonth'] = unseen_rows['SaleDate'].dt.month

# Drop the original SaleDate column since it's not numerical
unseen_rows = unseen_rows.drop(columns=['SaleDate'])

In [ ]:
# Numeric features
unseen_numeric_columns = ['NetWeight', 'ExchangeRate', 'SaleMonth', 'SaleYear']

In [ ]:
# categorical columns to encode
unseen_categorical_columns = ['ATCRegion', 'Grade', 'Elevation', 'processingMethod', 'SubRegion', 'SubElevation']

# one-hot encoding
unseen_encoded = pd.get_dummies(unseen_rows, columns=unseen_categorical_columns, drop_first=True)

In [ ]:
# Combine numeric and categorical features
features_unseen = unseen_numeric_columns + [col for col in df_encoded.columns if col not in unseen_numeric_columns + ['NetValue']]

In [ ]:
# Load the feature list from the file
with open('model_features.txt', 'r') as f:
    model_features = [line.strip() for line in f]

In [ ]:
X_unseen_encoded = unseen_encoded.reindex(columns=model_features, fill_value=0)

In [ ]:
X_unseen = X_unseen_encoded
y_unseen = unseen_rows['NetValue']

In [ ]:
import joblib
model = joblib.load('/content/random_forest_regression_model_V3.pkl')

In [ ]:
y_pred_unseen = model.predict(X_unseen_encoded)

In [ ]:
mae_unseen = mean_absolute_error(y_unseen, y_pred_unseen)
r2_unseen = r2_score(y_unseen, y_pred_unseen)
mape_unseen = mean_absolute_percentage_error(y_test_final, y_pred_rf_final)

print(f"MAE on unseen data: {mae_unseen}")
print(f"R² on unseen data: {r2_unseen}")
print(f"MAPE on unseen data: {mape_unseen}")

MAE on unseen data: 407.09041747309726
R² on unseen data: 0.9977030406012593
MAPE on unseen data: 0.0007869009210292356
